# Deal Desk Agent — Colab Enterprise Deploy

**FSI Deal Desk Pipeline — Anthropic + Google Cloud: Better Together**

This notebook stands up the entire Deal Desk Agent system in a fresh Google Cloud project, end-to-end:

1. Enables APIs, creates service accounts, sets IAM
2. Creates BigQuery dataset + tables, seeds synthetic FSI client data
3. Builds three container images via Cloud Build (backend, frontend, computer-use VM)
4. Deploys backend + frontend to Cloud Run, browser VM to Compute Engine
5. Stores secrets in Secret Manager (`AGENT_SECRET`, Salesforce credentials)
6. Deploys the ADK agent to Vertex AI Agent Engine — captures the resource ID
7. Registers the agent with **Gemini Enterprise** via the Discovery Engine API — uses the captured Agent Engine ID
8. Runs smoke tests
9. Provides teardown cells (last section — **do not skip if you want to stop billing**)

Total wall-clock: **25–40 minutes**. Most of that is Cloud Build + GCE cold-start + Agent Engine deploy.

---

> ## Disclaimer — Use At Your Own Risk
>
> This notebook deploys a **demonstration** system showcasing the art of the possible with Claude on Vertex AI, Google ADK, and Computer Use. It is **not production-ready** and is **not supported**.
>
> The deployed system is single-tenant, drives a Salesforce Developer Edition sandbox, uses in-memory ADK sessions, and ships with the known limitations documented in [`DESIGN.md` §9](../../03-demos/deal-desk-agent/DESIGN.md). Sample BigQuery data is synthetic and does not represent real clients.
>
> **Do not deploy this against real client data, production Salesforce orgs, or regulated workloads.** Adapting any part of this for production use requires (at minimum): IAM-based auth in place of the shared-secret pattern, persistent session storage, rate limiting, structured audit logging, secret rotation, network controls (VPC SC), and a full independent security review.
>
> Code is provided **as-is, without warranty of any kind**.

---

> ## Cost notice
>
> A full deploy + ~30 min of idle costs roughly **$5–15** in GCE + Cloud Run + Vertex inference. The GCE browser VM bills 24/7 if left running. **Run the teardown cells at the end of your session.**


## 1. Prerequisites — read before running any cell

The notebook can detect and fail-fast on most issues, but four things must be done **before** opening this notebook. Skip any of them and the preflight cell in §3 will stop with an error.

### 1.1 GCP project
- A GCP project with **billing enabled**
- You are an Owner or have these roles: `roles/serviceusage.serviceUsageAdmin`, `roles/iam.serviceAccountAdmin`, `roles/run.admin`, `roles/compute.admin`, `roles/artifactregistry.admin`, `roles/bigquery.admin`, `roles/aiplatform.admin`, `roles/secretmanager.admin`, `roles/storage.admin`, `roles/discoveryengine.admin`

### 1.2 Vertex AI Claude model access (most common silent failure)
For your project, you must have explicit access to all three Claude models. Console → **Vertex AI → Model Garden** → search "Claude" → for each, click **Enable** and accept Anthropic's commercial terms:

| Model ID | Used for |
|---|---|
| `claude-opus-4-5@20251101` | Research + Synthesis agents |
| `claude-sonnet-4-6@default` | Compliance + Salesforce (Computer Use) agents |
| `claude-haiku-4-5@20251001` | Risk scoring agent |

Region: **`us-east5`** (this is the only Claude region in the demo).

If skipped: every deploy succeeds, every inference returns 403. The preflight cell catches this.

### 1.3 Salesforce Developer Edition org

The browser VM container reads only `SALESFORCE_URL` — login is human-in-the-loop the first time you open noVNC, then Computer Use takes over. You need a fresh DE org.

1. Go to **https://developer.salesforce.com/signup**
2. Fill the form. Two field gotchas:
   - **Email** — must be one you can receive at; you'll click a verification link
   - **Username** — must be in *email format* (e.g., `dealdesk.demo@example.com`), but does NOT have to be a real address. It's just the login string. Pick something globally unique across all of Salesforce.
3. Submit → check email → click the verification link → set your password
4. After login you'll land at a URL like `https://orgfarm-XXXXXXXXXX-dev-ed.develop.lightning.force.com/lightning/page/home`. **Copy the host portion only** (everything before `/lightning/`) — that's your `SALESFORCE_URL` for §9.2.
5. **Do NOT enable MFA** on this dev org. If you accidentally enabled it, the Computer Use agent will get stuck on the verification step. Disable under Setup → Identity Verification.
6. **Do NOT** create a security token or Connected App — this demo uses interactive UI login via Computer Use, not API auth.

You will paste the URL, username (the email-format login string), and password into the cell in §9.2. They go into Secret Manager — never written to the notebook source.

### 1.4 Existing Gemini Enterprise engine

You need an existing **Gemini Enterprise / Agentspace engine** to register the deployed agent into. To find your engine ID:
- Console → **Gemini Enterprise → Apps** → click your app → the **App ID** (also called Engine ID) is shown in the URL: `.../engines/{ENGINE_ID}/...`
- Note it down for cell 18.1

If you don't have an engine yet, create one first via **Gemini Enterprise → Apps → Create App → Search**. The notebook does NOT create the engine for you (creating one provisions billable resources and is a deliberate decision).

### 1.5 Operator workstation
- A modern browser to open the noVNC URL and the deployed frontend
- Your **public IP** — the noVNC firewall rule restricts to your IP only. The notebook auto-detects via `curl ifconfig.me` and lets you override before creating the rule.


## 2. Configuration

Set everything you'll need in this single cell. Every later cell reads from these variables.

If you re-run the notebook later (after a partial failure or to update a single resource), only this cell and the preflight cell need to be re-run before jumping to the section you want.

In [ ]:
import os
import subprocess
import json

# ─── REQUIRED — change these ───
PROJECT_ID = "your-gcp-project-id"            # e.g., "my-deal-desk-demo"
GEMINI_ENTERPRISE_ENGINE_ID = "your-engine-id"  # from Console → Gemini Enterprise → Apps

# ─── Regions (pinned — do not change unless you know why) ───
MODEL_REGION = "us-east5"      # Vertex AI Claude region (only region in demo)
INFRA_REGION = "us-central1"   # Cloud Run + GCE + Artifact Registry
BQ_LOCATION  = "US"            # BigQuery multi-region
AGENT_ENGINE_LOCATION = "us-central1"
DISCOVERY_ENGINE_LOCATION = "global"  # Gemini Enterprise engines live in 'global' or 'eu'

# ─── Resource names (sane defaults — change if they collide with existing) ───
ARTIFACT_REPO   = "deal-desk-agent"
SERVICE_ACCOUNT = "deal-desk-agent-sa"
BACKEND_SERVICE  = "deal-desk-backend"
FRONTEND_SERVICE = "deal-desk-frontend"
GCE_INSTANCE     = "deal-desk-browser"
GCE_STATIC_IP    = "deal-desk-browser-ip"
GCE_ZONE         = f"{INFRA_REGION}-a"
GCE_MACHINE_TYPE = "n2-standard-4"      # upgraded from e2-standard-4 for Chrome+CU loop
BQ_DATASET = "deal_desk_agent"

# ─── Model IDs (pinned to what the agent code expects) ───
OPUS_MODEL   = "claude-opus-4-5@20251101"
SONNET_MODEL = "claude-sonnet-4-6@default"
HAIKU_MODEL  = "claude-haiku-4-5@20251001"

# ─── Secret names in Secret Manager ───
SECRET_AGENT      = "deal-desk-agent-secret"
SECRET_SF_URL     = "deal-desk-salesforce-url"
SECRET_SF_USER    = "deal-desk-salesforce-username"
SECRET_SF_PASS    = "deal-desk-salesforce-password"

# ─── Operator public IP (for noVNC firewall) — auto-detected, override below if needed ───
try:
    OPERATOR_PUBLIC_IP = subprocess.check_output(
        ["curl", "-s", "--max-time", "5", "https://ifconfig.me"]
    ).decode().strip()
except Exception:
    OPERATOR_PUBLIC_IP = ""

# Manual override — uncomment and set if auto-detection wrong (e.g., behind VPN, shared egress)
# OPERATOR_PUBLIC_IP = "203.0.113.42"

# ─── Derived / computed (do not edit) ───
SA_EMAIL = f"{SERVICE_ACCOUNT}@{PROJECT_ID}.iam.gserviceaccount.com"
AR_PREFIX = f"{INFRA_REGION}-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REPO}"
BACKEND_IMAGE  = f"{AR_PREFIX}/backend:latest"
FRONTEND_IMAGE = f"{AR_PREFIX}/frontend:latest"
BROWSER_IMAGE  = f"{AR_PREFIX}/browser-vm:latest"
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"

# Validate required values are set
assert PROJECT_ID and PROJECT_ID != "your-gcp-project-id", "Set PROJECT_ID before running."
assert GEMINI_ENTERPRISE_ENGINE_ID and GEMINI_ENTERPRISE_ENGINE_ID != "your-engine-id", \
    "Set GEMINI_ENTERPRISE_ENGINE_ID before running. See §1.4 for how to find it."
assert OPERATOR_PUBLIC_IP, "Could not auto-detect public IP. Set OPERATOR_PUBLIC_IP manually."

# Export to environment so subprocess gcloud calls inherit
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

print("Configuration loaded.")
print(f"  Project:           {PROJECT_ID}")
print(f"  Gemini Engine ID:  {GEMINI_ENTERPRISE_ENGINE_ID}")
print(f"  Model region:      {MODEL_REGION}")
print(f"  Infra region:      {INFRA_REGION}")
print(f"  Operator IP:       {OPERATOR_PUBLIC_IP}")
print(f"  Service account:   {SA_EMAIL}")
print(f"  Staging bucket:    {STAGING_BUCKET}")


## 3. Preflight checks — fail fast before any deploy

This cell makes ~7 quick checks against the configured project. If any fail, it stops with a clear error and a remediation hint. Do not skip — the most common deploy failure is missing Vertex Claude model access, which produces silent 403s much later in the flow.

In [ ]:
import shlex
import urllib.request

def sh(cmd, capture=True, check=True):
    """Run a shell command and return stdout."""
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    result = subprocess.run(cmd, capture_output=capture, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{result.stderr}")
    return result.stdout.strip()

print("Running preflight checks...\n")

# 1. gcloud is configured to the right project
current = sh(f"gcloud config get-value project")
if current != PROJECT_ID:
    print(f"  Setting gcloud project to {PROJECT_ID}")
    sh(f"gcloud config set project {PROJECT_ID}")
print(f"  [OK] gcloud project = {PROJECT_ID}")

# 2. Billing is enabled
billing = sh(f"gcloud billing projects describe {PROJECT_ID} --format=value(billingEnabled)", check=False)
assert billing == "True", (
    f"Billing not enabled on project {PROJECT_ID}. "
    f"Enable at https://console.cloud.google.com/billing/linkedaccount?project={PROJECT_ID}"
)
print(f"  [OK] Billing enabled")

# 3. ADC works
import google.auth
try:
    creds, _ = google.auth.default()
    print(f"  [OK] Application Default Credentials available")
except Exception as e:
    raise RuntimeError(
        "ADC not configured. In Colab Enterprise this is normally automatic; "
        "if running locally run: gcloud auth application-default login"
    ) from e

# 4. Vertex AI Claude model access — try a 1-token rawPredict for each model
import google.auth.transport.requests
creds.refresh(google.auth.transport.requests.Request())
token = creds.token

for model in [OPUS_MODEL, SONNET_MODEL, HAIKU_MODEL]:
    url = (
        f"https://{MODEL_REGION}-aiplatform.googleapis.com/v1/"
        f"projects/{PROJECT_ID}/locations/{MODEL_REGION}/"
        f"publishers/anthropic/models/{model}:rawPredict"
    )
    body = json.dumps({
        "anthropic_version": "vertex-2023-10-16",
        "max_tokens": 1,
        "messages": [{"role": "user", "content": "hi"}],
    }).encode()
    req = urllib.request.Request(
        url, data=body,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            assert resp.status == 200
        print(f"  [OK] Vertex Claude {model} reachable")
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Vertex Claude {model} returned HTTP {e.code}.\n"
            f"Response: {body[:500]}\n"
            f"Most likely cause: model not enabled for this project. "
            f"Enable at https://console.cloud.google.com/vertex-ai/model-garden?project={PROJECT_ID} "
            f"(search 'Claude', click each model, Enable, accept terms)."
        ) from None

# 5. Operator IP is a sane IPv4
import ipaddress
try:
    ipaddress.IPv4Address(OPERATOR_PUBLIC_IP)
    print(f"  [OK] Operator public IP = {OPERATOR_PUBLIC_IP}")
except ValueError:
    raise RuntimeError(
        f"OPERATOR_PUBLIC_IP={OPERATOR_PUBLIC_IP!r} is not a valid IPv4. "
        f"Set it manually in §2."
    )

# 6. Discoveryengine API reachable (we'll enable it next, just check the project exists for it)
print(f"  [OK] All preflight checks passed.\n")
print("Ready to deploy. Continue to §4.")


## 4. Enable APIs

Enables every API the deploy needs. Idempotent — safe to re-run.

In [ ]:
APIS = [
    "aiplatform.googleapis.com",
    "bigquery.googleapis.com",
    "run.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "compute.googleapis.com",
    "secretmanager.googleapis.com",
    "iamcredentials.googleapis.com",
    "discoveryengine.googleapis.com",
    "storage.googleapis.com",
    "cloudresourcemanager.googleapis.com",
]

print(f"Enabling {len(APIS)} APIs (this can take 1-2 minutes)...")
sh(f"gcloud services enable {' '.join(APIS)} --project={PROJECT_ID}")
print("All APIs enabled.")


## 5. Install pinned Python tooling

Pinned versions matching `backend/requirements.txt` and `agent_deploy/requirements.txt` from the repo, so the Agent Engine deploy and the inline checks behave the same as the deployed services.

In [ ]:
%pip install -q --upgrade \
    "google-cloud-aiplatform[agent_engines,adk]>=1.78.0" \
    "google-adk>=1.2.0" \
    "anthropic[vertex]>=0.43.0" \
    "google-cloud-bigquery>=3.27.0" \
    "google-cloud-secret-manager>=2.20.0" \
    "google-cloud-discoveryengine>=0.13.0" \
    "google-cloud-storage>=2.18.0" \
    "httpx>=0.27.0" \
    "cloudpickle>=3.0.0" \
    "pydantic>=2.0.0"

print("\nIMPORTANT: if Colab prompted to RESTART the runtime, restart now then re-run §2 + §3 + §5.")


## 6. Clone the demo repo

Pulls the Deal Desk source from the partner repo. The Agent Engine deploy and Cloud Build cells reference these files directly.

In [ ]:
REPO_URL = "https://github.com/PTA-Co-innovation-Team/Anthropic-Google-Co-Innovation.git"
WORKDIR = "/content/anthropic-google-coinnov"
DEMO_DIR = f"{WORKDIR}/03-demos/deal-desk-agent"

if not os.path.isdir(WORKDIR):
    sh(f"git clone --depth 1 {REPO_URL} {WORKDIR}")
else:
    print(f"Repo already cloned at {WORKDIR} — pulling latest")
    sh(f"git -C {WORKDIR} pull --ff-only")

assert os.path.isdir(DEMO_DIR), f"Expected demo at {DEMO_DIR} — did the repo structure change?"
print(f"\nDemo source ready at: {DEMO_DIR}")
print("Tree:")
sh(f"ls -la {DEMO_DIR}", capture=False)


## 7. BigQuery — dataset, schemas, and synthetic seed data

The agent reads from four tables. Schemas are derived from `backend/tools/bigquery_tools.py` (every read tool dictates what columns it expects).

The repo ships **no seed data**. This section seeds ~20 synthetic FSI clients (pension funds, family offices, RIAs, hedge funds) with matching market intel and compliance records. Names are fictional. Numbers are realistic but invented.

### 7.1 Create dataset

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
dataset_ref = f"{PROJECT_ID}.{BQ_DATASET}"

dataset = bigquery.Dataset(dataset_ref)
dataset.location = BQ_LOCATION
try:
    dataset = bq.create_dataset(dataset, exists_ok=True)
    print(f"Dataset ready: {dataset_ref} ({BQ_LOCATION})")
except Exception as e:
    raise RuntimeError(f"Failed to create dataset {dataset_ref}: {e}")


### 7.2 Create tables

Schemas match what `query_client_data`, `query_market_intelligence`, `query_compliance_records`, and `insert_deal_package` expect.

In [ ]:
SCHEMAS = {
    "clients": [
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("aum_millions", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("strategy", "STRING"),
        bigquery.SchemaField("mandate_type", "STRING"),
        bigquery.SchemaField("fee_structure", "STRING"),
        bigquery.SchemaField("primary_contact", "STRING"),
        bigquery.SchemaField("primary_contact_title", "STRING"),
        bigquery.SchemaField("primary_contact_email", "STRING"),
        bigquery.SchemaField("relationship_status", "STRING"),
        bigquery.SchemaField("domicile", "STRING"),
        bigquery.SchemaField("founded_year", "INT64"),
        bigquery.SchemaField("client_type", "STRING"),
    ],
    "market_intelligence": [
        bigquery.SchemaField("client_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("date", "DATE", mode="REQUIRED"),
        bigquery.SchemaField("record_type", "STRING"),  # SEC_FILING | NEWS | MARKET_DATA
        bigquery.SchemaField("title", "STRING"),
        bigquery.SchemaField("source", "STRING"),
        bigquery.SchemaField("summary", "STRING"),
        bigquery.SchemaField("relevance_score", "FLOAT64"),
    ],
    "compliance_records": [
        bigquery.SchemaField("client_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("kyc_status", "STRING"),       # PASSED | PENDING | FAILED
        bigquery.SchemaField("aml_status", "STRING"),
        bigquery.SchemaField("sanctions_status", "STRING"), # CLEAR | HIT | REVIEW
        bigquery.SchemaField("finra_status", "STRING"),     # REGISTERED | NOT_REQUIRED | PENDING
        bigquery.SchemaField("risk_tier", "STRING"),        # LOW | MEDIUM | HIGH
        bigquery.SchemaField("last_review_date", "DATE"),
        bigquery.SchemaField("notes", "STRING"),
    ],
    "deal_packages": [
        bigquery.SchemaField("deal_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("client_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("aum_millions", "FLOAT64"),
        bigquery.SchemaField("strategy", "STRING"),
        bigquery.SchemaField("mandate_type", "STRING"),
        bigquery.SchemaField("fee_structure", "STRING"),
        bigquery.SchemaField("compliance_status", "STRING"),
        bigquery.SchemaField("risk_tier", "STRING"),
        bigquery.SchemaField("risk_score", "FLOAT64"),
        bigquery.SchemaField("primary_contact", "STRING"),
        bigquery.SchemaField("primary_contact_title", "STRING"),
        bigquery.SchemaField("salesforce_opportunity_id", "STRING"),
        bigquery.SchemaField("status", "STRING"),
        bigquery.SchemaField("created_by", "STRING"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
        bigquery.SchemaField("notes", "STRING"),
    ],
}

for table_name, schema in SCHEMAS.items():
    table_ref = f"{dataset_ref}.{table_name}"
    table = bigquery.Table(table_ref, schema=schema)
    bq.create_table(table, exists_ok=True)
    print(f"  Table ready: {table_name}")
print("\nAll 4 tables created.")


### 7.3 Seed synthetic data

20 fictional FSI clients spanning hedge funds, pension funds, family offices, and RIAs. Each has matching market intelligence (SEC filings, news) and compliance records. Deterministic — re-running this cell against an empty table produces the same data.

If the tables already have data from a previous run, this cell **truncates and reloads** to keep things deterministic.

In [ ]:
from datetime import date, timedelta
import random

# Deterministic seed so re-runs produce identical data
random.seed(42)

CLIENTS = [
    # (name, aum_millions, strategy, mandate_type, fee_structure, contact, title, email, status, domicile, founded, client_type)
    ("Beacon Hedge Fund",            500.0, "Global Macro",            "Discretionary",      "2/20",     "Marcus Webb",      "CIO",              "mwebb@beaconhf.com",       "Active",    "Cayman Islands", 2008, "Hedge Fund"),
    ("ACME Capital Management",      250.0, "Long/Short Equity",       "Institutional",      "2/20",     "Sarah Chen",       "CIO",              "schen@acmecap.com",        "Prospect",  "Delaware",       2014, "Hedge Fund"),
    ("Meridian Partners",            180.0, "Multi-Strategy",          "Separately Managed", "1.5/15",   "James Holloway",   "Managing Partner", "jholloway@meridian.com",   "Returning", "Connecticut",    2011, "Hedge Fund"),
    ("Cascade Pension Trust",       4200.0, "Liability-Driven",        "Institutional",      "0.35 flat","Diane Park",       "Treasurer",        "dpark@cascadepension.org", "Active",    "Oregon",         1984, "Pension Fund"),
    ("Northstar Endowment",         2800.0, "60/40 Balanced",          "Institutional",      "0.40 flat","Robert Kim",       "CIO",              "rkim@northstar.edu",       "Active",    "Massachusetts",  1962, "Endowment"),
    ("Hartwell Family Office",       320.0, "Multi-Asset",             "Wealth Management",  "1.0 flat", "Eleanor Hartwell", "Principal",        "ehartwell@hartwellfo.com", "Returning", "New York",       1998, "Family Office"),
    ("Apex Quantitative",            850.0, "Statistical Arbitrage",   "Discretionary",      "2/30",     "Vikram Rao",       "Head of Research", "vrao@apexquant.com",       "Active",    "Delaware",       2016, "Hedge Fund"),
    ("Greylock Wealth Advisors",     145.0, "Tax-Aware Equity",        "Wealth Management",  "0.85 flat","Patricia Mendez",  "Managing Director","pmendez@greylockwa.com",   "Active",    "California",     2009, "RIA"),
    ("Summit Insurance Holdings",   3500.0, "Investment Grade Credit", "Institutional",      "0.30 flat","Hideki Tanaka",    "CIO",              "htanaka@summitins.com",    "Active",    "Bermuda",        1995, "Insurance"),
    ("Larkspur Venture Partners",    220.0, "Late-Stage Venture",      "Limited Partnership","2/25",     "Alice Donovan",    "General Partner",  "adonovan@larkspurvp.com",  "Prospect",  "Delaware",       2019, "VC Fund"),
    ("Whitestone Macro Fund",        680.0, "Discretionary Macro",     "Discretionary",      "1.75/20",  "Niall O'Connor",   "Founder",          "noconnor@whitestone.com",  "Returning", "Cayman Islands", 2013, "Hedge Fund"),
    ("Cordillera Mining Pension",   1900.0, "Real Assets Tilt",        "Institutional",      "0.45 flat","Maria Aguilar",    "Investment Officer","maguilar@cordmining.org", "Active",    "Colorado",       1976, "Pension Fund"),
    ("Bristol Sovereign Capital",   8500.0, "Sovereign Wealth",        "Institutional",      "0.20 flat","Khalid Al-Rashid", "Director, Public Markets","kalrashid@bristolsc.gov","Active","Singapore",      2007, "Sovereign Wealth"),
    ("Driftwood Asset Mgmt",         410.0, "Long-Only Equity",        "Mutual Fund Sub-Advisor","0.65 flat","Henrik Olsen", "Lead PM",          "holsen@driftwoodam.com",   "Returning", "Minnesota",      2003, "Asset Manager"),
    ("Provident Healthcare Trust",  1200.0, "ESG-Tilted Balanced",     "Institutional",      "0.50 flat","Sandra Whittaker", "CIO",              "swhittaker@providenthc.org","Active","Pennsylvania",   1989, "Healthcare System"),
    ("Riverbend Credit Opportunities",560.0, "Distressed Credit",      "Closed-End Fund",    "1.5/20",   "Tomas Rivera",     "Portfolio Manager","trivera@riverbendco.com",  "Active",    "Cayman Islands", 2012, "Credit Fund"),
    ("Sentinel Multi-Family Office", 290.0, "Customized Multi-Asset",  "Wealth Management",  "0.95 flat","Hannah Goldberg",  "Lead Advisor",     "hgoldberg@sentinelmfo.com","Returning", "Illinois",       2006, "Family Office"),
    ("Cobalt Energy Partners",       780.0, "Energy Infrastructure",   "Limited Partnership","2/20",     "Wesley Carrington","Operating Partner","wcarrington@cobaltenergy.com","Prospect","Texas",         2015, "Private Equity"),
    ("Olympia Foundation",           640.0, "Endowment Style",         "Institutional",      "0.42 flat","Frances Liang",    "Director of Investments","fliang@olympiafdn.org", "Active",  "Washington",     1971, "Foundation"),
    ("Tidewater RIA Group",           95.0, "Core/Satellite",          "Wealth Management",  "0.75 flat","David Pham",       "Founder",          "dpham@tidewaterria.com",   "Returning", "Virginia",       2017, "RIA"),
]

# Truncate then load (idempotent)
print("Loading clients table...")
sh(f"bq query --project_id={PROJECT_ID} --use_legacy_sql=false 'TRUNCATE TABLE `{PROJECT_ID}.{BQ_DATASET}.clients`'", check=False)

rows = [
    {
        "name": c[0], "aum_millions": c[1], "strategy": c[2], "mandate_type": c[3],
        "fee_structure": c[4], "primary_contact": c[5], "primary_contact_title": c[6],
        "primary_contact_email": c[7], "relationship_status": c[8],
        "domicile": c[9], "founded_year": c[10], "client_type": c[11],
    }
    for c in CLIENTS
]
errors = bq.insert_rows_json(f"{dataset_ref}.clients", rows)
assert not errors, f"Failed to insert clients: {errors}"
print(f"  Inserted {len(rows)} client rows")

# Market intelligence — 1-3 records per client
print("\nLoading market_intelligence table...")
sh(f"bq query --project_id={PROJECT_ID} --use_legacy_sql=false 'TRUNCATE TABLE `{PROJECT_ID}.{BQ_DATASET}.market_intelligence`'", check=False)

INTEL_TEMPLATES = {
    "SEC_FILING": [
        ("Form ADV Part 2A annual update", "SEC EDGAR", "Annual brochure update; AUM and strategy disclosures consistent with prior year."),
        ("Form 13F Q3 holdings disclosure", "SEC EDGAR", "Reportable positions filed; concentration in financials and tech sectors."),
        ("Form D — exempt offering", "SEC EDGAR", "Notice of new private fund launch under Reg D 506(c)."),
    ],
    "NEWS": [
        ("Quarterly letter to investors", "Pitchbook", "Returns in line with peer median; manager attributes outperformance to defensive positioning."),
        ("Senior hire announcement", "Bloomberg", "Hired new head of risk from a tier-1 bank; signals institutional process maturation."),
        ("Strategy expansion into private credit", "Reuters", "Plans to launch a sleeve dedicated to direct lending in 2026."),
        ("Closed Series III fund at hard cap", "WSJ", "Oversubscribed close at $750M hard cap; LPs include public pensions."),
    ],
    "MARKET_DATA": [
        ("Trailing 12mo Sharpe ratio", "Internal estimate", "Sharpe ~1.2 vs peer median 0.9; volatility within mandate band."),
        ("Drawdown vs benchmark", "Bloomberg", "Max drawdown 8.4% in last 24 months vs benchmark 11.2%."),
    ],
}

intel_rows = []
today = date.today()
for c in CLIENTS:
    name = c[0]
    n_records = random.randint(1, 3)
    for i in range(n_records):
        record_type = random.choice(list(INTEL_TEMPLATES.keys()))
        tmpl = random.choice(INTEL_TEMPLATES[record_type])
        days_ago = random.randint(7, 180)
        intel_rows.append({
            "client_name": name,
            "date": (today - timedelta(days=days_ago)).isoformat(),
            "record_type": record_type,
            "title": tmpl[0],
            "source": tmpl[1],
            "summary": tmpl[2],
            "relevance_score": round(random.uniform(0.55, 0.98), 2),
        })

errors = bq.insert_rows_json(f"{dataset_ref}.market_intelligence", intel_rows)
assert not errors, f"Failed to insert market_intelligence: {errors}"
print(f"  Inserted {len(intel_rows)} market intelligence rows")

# Compliance — one record per client; bias most to PASSED with a few REVIEW/PENDING
print("\nLoading compliance_records table...")
sh(f"bq query --project_id={PROJECT_ID} --use_legacy_sql=false 'TRUNCATE TABLE `{PROJECT_ID}.{BQ_DATASET}.compliance_records`'", check=False)

def _comp_status(client_type, idx):
    # Most clients are clean; sprinkle a handful of REVIEW/PENDING for demo realism
    if idx in (1, 9, 17):  # ACME (prospect), Larkspur, Cobalt
        return ("PENDING", "PENDING", "REVIEW", "PENDING", "MEDIUM", "Awaiting completed KYC packet from client; sanctions hit on unrelated namesake — manual review in progress.")
    if idx == 4:  # Northstar — endowment, large; clean but elevated risk tier from concentration
        return ("PASSED", "PASSED", "CLEAR", "NOT_REQUIRED", "MEDIUM", "Concentration risk in single-manager allocations; periodic IC review.")
    if client_type in ("Sovereign Wealth", "Pension Fund", "Insurance", "Foundation", "Endowment"):
        return ("PASSED", "PASSED", "CLEAR", "NOT_REQUIRED", "LOW", "Institutional client; standard institutional onboarding completed.")
    return ("PASSED", "PASSED", "CLEAR", "REGISTERED", random.choice(["LOW", "MEDIUM"]), "Standard FINRA-registered intermediary; routine annual review.")

comp_rows = []
for idx, c in enumerate(CLIENTS):
    name, _, _, _, _, _, _, _, _, _, _, ctype = c
    kyc, aml, sanc, finra, risk, notes = _comp_status(ctype, idx)
    comp_rows.append({
        "client_name": name,
        "kyc_status": kyc,
        "aml_status": aml,
        "sanctions_status": sanc,
        "finra_status": finra,
        "risk_tier": risk,
        "last_review_date": (today - timedelta(days=random.randint(15, 200))).isoformat(),
        "notes": notes,
    })
errors = bq.insert_rows_json(f"{dataset_ref}.compliance_records", comp_rows)
assert not errors, f"Failed to insert compliance_records: {errors}"
print(f"  Inserted {len(comp_rows)} compliance rows")

# deal_packages stays empty — populated by the agent at runtime
print("\nSeed complete:")
counts = list(bq.query(f"""
    SELECT 'clients' AS t, COUNT(*) AS n FROM `{dataset_ref}.clients`
    UNION ALL SELECT 'market_intelligence', COUNT(*) FROM `{dataset_ref}.market_intelligence`
    UNION ALL SELECT 'compliance_records', COUNT(*) FROM `{dataset_ref}.compliance_records`
    UNION ALL SELECT 'deal_packages', COUNT(*) FROM `{dataset_ref}.deal_packages`
""").result())
for r in counts:
    print(f"  {r.t:25s} {r.n:>5d} rows")


## 8. Service account + IAM

Single service account used by Cloud Run (backend), GCE (browser VM), and Agent Engine. Bound to the minimum roles needed.

In [ ]:
# Create SA (idempotent)
existing = sh(f"gcloud iam service-accounts list --filter=email:{SA_EMAIL} --format=value(email)", check=False)
if existing != SA_EMAIL:
    sh(f"gcloud iam service-accounts create {SERVICE_ACCOUNT} "
       f"--display-name='Deal Desk Agent SA' --project={PROJECT_ID}")
    print(f"  Created service account: {SA_EMAIL}")
else:
    print(f"  Service account already exists: {SA_EMAIL}")

ROLES = [
    "roles/aiplatform.user",
    "roles/bigquery.dataEditor",
    "roles/bigquery.jobUser",
    "roles/secretmanager.secretAccessor",
    "roles/storage.objectViewer",
    "roles/storage.objectCreator",  # for Agent Engine staging bucket
    "roles/logging.logWriter",
    "roles/monitoring.metricWriter",
]

print(f"\nBinding {len(ROLES)} roles...")
for role in ROLES:
    sh(f"gcloud projects add-iam-policy-binding {PROJECT_ID} "
       f"--member=serviceAccount:{SA_EMAIL} --role={role} --condition=None --quiet",
       check=False)
print("IAM bindings applied.")


## 9. Secrets — Secret Manager

### 9.1 Generate AGENT_SECRET

The shared secret authenticating Cloud Run backend → GCE browser VM. 32 hex bytes. Generated here, stored in Secret Manager, and read by both the backend and the VM at deploy time.

In [ ]:
import secrets as _secrets

def upsert_secret(name: str, value: str):
    """Create or update a secret in Secret Manager."""
    # Create the secret container (if missing)
    sh(f"gcloud secrets describe {name} --project={PROJECT_ID}", check=False)
    exists = sh(f"gcloud secrets list --filter=name~{name}$ --format=value(name) --project={PROJECT_ID}", check=False)
    if name not in exists:
        sh(f"gcloud secrets create {name} --replication-policy=automatic --project={PROJECT_ID}")
        print(f"  Created secret: {name}")
    # Add a new version with the provided value
    p = subprocess.run(
        ["gcloud", "secrets", "versions", "add", name,
         f"--project={PROJECT_ID}", "--data-file=-"],
        input=value, text=True, capture_output=True, check=True,
    )
    print(f"  Added version to {name}")

AGENT_SECRET_VALUE = _secrets.token_hex(32)
upsert_secret(SECRET_AGENT, AGENT_SECRET_VALUE)
print("\nAGENT_SECRET stored. The plaintext value is held only in this notebook variable for the rest of the deploy.")


### 9.2 Salesforce credentials

Paste the URL, username, and password from the DE org you signed up for in §1.3.

The password input uses `getpass` so it is NOT echoed to the notebook. Once stored in Secret Manager you can clear the cell output if you want.

In [ ]:
from getpass import getpass

print("Paste your Salesforce Developer Edition details (from §1.3):\n")
SF_URL  = input("SALESFORCE_URL (e.g., https://orgfarm-XXX-dev-ed.develop.lightning.force.com): ").strip()
SF_USER = input("Salesforce username (the email-format login): ").strip()
SF_PASS = getpass("Salesforce password: ")

assert SF_URL.startswith("https://") and "lightning.force.com" in SF_URL, \
    "URL should look like https://orgfarm-XXX-dev-ed.develop.lightning.force.com"
assert "@" in SF_USER, "Username should be email-format (Salesforce convention)"
assert len(SF_PASS) >= 8, "Password seems too short"

upsert_secret(SECRET_SF_URL,  SF_URL)
upsert_secret(SECRET_SF_USER, SF_USER)
upsert_secret(SECRET_SF_PASS, SF_PASS)

print("\nAll three Salesforce secrets stored. Clear this cell's output if you want.")


## 10. Artifact Registry

Single Docker repo for all three images.

In [ ]:
existing = sh(f"gcloud artifacts repositories list --location={INFRA_REGION} "
              f"--filter=name~/{ARTIFACT_REPO}$ --format=value(name)", check=False)
if ARTIFACT_REPO not in existing:
    sh(f"gcloud artifacts repositories create {ARTIFACT_REPO} "
       f"--repository-format=docker --location={INFRA_REGION} "
       f"--description='Deal Desk Agent images' --project={PROJECT_ID}")
    print(f"  Created repo: {ARTIFACT_REPO}")
else:
    print(f"  Repo already exists: {ARTIFACT_REPO}")


## 11. Build container images via Cloud Build

Three images, sequential, ~5 min each. Cloud Build replaces local `docker build` since Colab Enterprise runtimes have no Docker daemon.

If a build fails midway, re-run only that cell — Cloud Build is idempotent.

### 11.1 Backend (FastAPI on Cloud Run)

In [ ]:
sh(f"cd {DEMO_DIR}/backend && gcloud builds submit "
   f"--tag={BACKEND_IMAGE} --region={INFRA_REGION} --project={PROJECT_ID} --quiet",
   capture=False)
print("\nBackend image built.")


### 11.2 Frontend (React/Vite on Cloud Run, runtime config injection)

In [ ]:
sh(f"cd {DEMO_DIR}/frontend && gcloud builds submit "
   f"--tag={FRONTEND_IMAGE} --region={INFRA_REGION} --project={PROJECT_ID} --quiet",
   capture=False)
print("\nFrontend image built.")


### 11.3 Computer Use VM (Ubuntu + Xvfb + Chrome + noVNC + agent server)

In [ ]:
sh(f"cd {DEMO_DIR}/computer-use && gcloud builds submit "
   f"--tag={BROWSER_IMAGE} --region={INFRA_REGION} --project={PROJECT_ID} --quiet",
   capture=False)
print("\nComputer Use VM image built.")


## 12. Deploy the browser VM (GCE)

### 12.1 Reserve a static external IP

In [ ]:
existing = sh(f"gcloud compute addresses list --regions={INFRA_REGION} "
              f"--filter=name:{GCE_STATIC_IP} --format=value(name)", check=False)
if GCE_STATIC_IP not in existing:
    sh(f"gcloud compute addresses create {GCE_STATIC_IP} --region={INFRA_REGION} --project={PROJECT_ID}")

VM_STATIC_IP = sh(f"gcloud compute addresses describe {GCE_STATIC_IP} "
                  f"--region={INFRA_REGION} --format=value(address) --project={PROJECT_ID}")
print(f"Static IP: {VM_STATIC_IP}")

# Derived URLs
NOVNC_URL_VAL  = f"http://{VM_STATIC_IP}:6080/vnc.html?autoconnect=true&resize=scale"
BROWSER_AGENT_URL = f"http://{VM_STATIC_IP}:8090"


### 12.2 Firewall rules

Two rules:
- `allow-novnc-{PROJECT_ID}` — tcp:6080 restricted to **your operator IP only** (no app-layer auth on noVNC)
- `allow-agent-api-{PROJECT_ID}` — tcp:8090 open to 0.0.0.0/0 because Cloud Run's egress IPs are dynamic; the API is gated by `X-Agent-Secret`

In [ ]:
NOVNC_RULE = f"allow-novnc-{PROJECT_ID[:25]}"
AGENT_RULE = f"allow-agent-api-{PROJECT_ID[:23]}"

# noVNC — operator IP only
sh(f"gcloud compute firewall-rules describe {NOVNC_RULE} --project={PROJECT_ID}", check=False)
existing = sh(f"gcloud compute firewall-rules list --filter=name:{NOVNC_RULE} --format=value(name) --project={PROJECT_ID}", check=False)
if NOVNC_RULE not in existing:
    sh(f"gcloud compute firewall-rules create {NOVNC_RULE} "
       f"--direction=INGRESS --action=ALLOW --rules=tcp:6080 "
       f"--source-ranges={OPERATOR_PUBLIC_IP}/32 "
       f"--target-tags=deal-desk-browser --project={PROJECT_ID}")
    print(f"  Created firewall rule {NOVNC_RULE} (source: {OPERATOR_PUBLIC_IP}/32)")
else:
    print(f"  Firewall rule {NOVNC_RULE} already exists")

# Agent API — open + header-gated
existing = sh(f"gcloud compute firewall-rules list --filter=name:{AGENT_RULE} --format=value(name) --project={PROJECT_ID}", check=False)
if AGENT_RULE not in existing:
    sh(f"gcloud compute firewall-rules create {AGENT_RULE} "
       f"--direction=INGRESS --action=ALLOW --rules=tcp:8090 "
       f"--source-ranges=0.0.0.0/0 "
       f"--target-tags=deal-desk-browser --project={PROJECT_ID}")
    print(f"  Created firewall rule {AGENT_RULE} (source: 0.0.0.0/0, header-gated)")
else:
    print(f"  Firewall rule {AGENT_RULE} already exists")


### 12.3 Create the GCE instance with the container

Container is invoked with `AGENT_SECRET` and `SALESFORCE_URL` injected from Secret Manager via metadata. Wait for instance RUNNING + container healthy (~3-5 min).

In [ ]:
import time

# Pull the salesforce URL we just stored — needed at container start so Chrome lands on the login page
SF_URL_VAL = sh(f"gcloud secrets versions access latest --secret={SECRET_SF_URL} --project={PROJECT_ID}")

# Idempotent create — delete first if it exists to apply latest config
existing = sh(f"gcloud compute instances list --zones={GCE_ZONE} "
              f"--filter=name:{GCE_INSTANCE} --format=value(name)", check=False)
if GCE_INSTANCE in existing:
    print(f"  Instance {GCE_INSTANCE} exists — deleting to redeploy with current config...")
    sh(f"gcloud compute instances delete {GCE_INSTANCE} --zone={GCE_ZONE} "
       f"--quiet --project={PROJECT_ID}")

sh(f"gcloud compute instances create-with-container {GCE_INSTANCE} "
   f"--zone={GCE_ZONE} --machine-type={GCE_MACHINE_TYPE} "
   f"--container-image={BROWSER_IMAGE} "
   f"--container-env=AGENT_SECRET={AGENT_SECRET_VALUE},SALESFORCE_URL={SF_URL_VAL},PROJECT_ID={PROJECT_ID},REGION={MODEL_REGION},SONNET_MODEL={SONNET_MODEL} "
   f"--tags=deal-desk-browser "
   f"--address={VM_STATIC_IP} "
   f"--service-account={SA_EMAIL} "
   f"--scopes=cloud-platform "
   f"--project={PROJECT_ID}", capture=False)

print(f"\nInstance created. Waiting for RUNNING state...")
for _ in range(60):
    status = sh(f"gcloud compute instances describe {GCE_INSTANCE} --zone={GCE_ZONE} "
               f"--format=value(status) --project={PROJECT_ID}", check=False)
    if status == "RUNNING":
        print(f"  Instance is RUNNING")
        break
    time.sleep(5)
else:
    raise RuntimeError(f"Instance never reached RUNNING; last status: {status}")

# Now wait for container/agent server to come up — Chrome+Xvfb startup takes ~60-90s
print("\nWaiting for agent server health endpoint to respond (~90s)...")
import urllib.request, urllib.error
for i in range(36):
    try:
        with urllib.request.urlopen(f"{BROWSER_AGENT_URL}/health", timeout=3) as r:
            if r.status == 200:
                print(f"  Agent server is healthy: {BROWSER_AGENT_URL}/health")
                break
    except (urllib.error.URLError, urllib.error.HTTPError, ConnectionError):
        pass
    time.sleep(5)
else:
    print(f"  WARNING: agent server didn't respond within 3 minutes. Check logs:")
    print(f"    gcloud compute instances get-serial-port-output {GCE_INSTANCE} --zone={GCE_ZONE}")


## 13. Deploy backend to Cloud Run

`AGENT_SECRET` injected from Secret Manager (best practice — even though we have it in memory, the deploy command should reference Secret Manager so secret rotation works without notebook re-runs).

In [ ]:
# Allow the backend SA to read the AGENT_SECRET (already granted secretAccessor at project level, but tighten just in case)
sh(f"gcloud secrets add-iam-policy-binding {SECRET_AGENT} "
   f"--member=serviceAccount:{SA_EMAIL} --role=roles/secretmanager.secretAccessor "
   f"--project={PROJECT_ID} --quiet", check=False)

env_vars = ",".join([
    f"PROJECT_ID={PROJECT_ID}",
    f"REGION={MODEL_REGION}",
    f"BQ_DATASET={BQ_DATASET}",
    f"MODEL_PROVIDER=claude",
    f"OPUS_MODEL={OPUS_MODEL}",
    f"SONNET_MODEL={SONNET_MODEL}",
    f"HAIKU_MODEL={HAIKU_MODEL}",
    f"BROWSER_AGENT_URL={BROWSER_AGENT_URL}",
])

sh(f"gcloud run deploy {BACKEND_SERVICE} "
   f"--image={BACKEND_IMAGE} --region={INFRA_REGION} "
   f"--service-account={SA_EMAIL} "
   f"--set-env-vars={env_vars} "
   f"--set-secrets=AGENT_SECRET={SECRET_AGENT}:latest "
   f"--memory=2Gi --cpu=2 --timeout=300 --max-instances=5 "
   f"--allow-unauthenticated --quiet --project={PROJECT_ID}", capture=False)

BACKEND_URL = sh(f"gcloud run services describe {BACKEND_SERVICE} "
                 f"--region={INFRA_REGION} --format=value(status.url) --project={PROJECT_ID}")
print(f"\nBackend URL: {BACKEND_URL}")

# Now redeploy with A2A_AGENT_URL set to the actual URL we just received,
# so the /.well-known/agent.json card returns a stable URL for Gemini Enterprise registration.
print("\nRedeploying backend with A2A_AGENT_URL set (so the agent card is stable)...")
env_vars_with_a2a = env_vars + f",A2A_AGENT_URL={BACKEND_URL}"
sh(f"gcloud run services update {BACKEND_SERVICE} "
   f"--region={INFRA_REGION} "
   f"--update-env-vars=A2A_AGENT_URL={BACKEND_URL} "
   f"--project={PROJECT_ID} --quiet", capture=False)
print(f"A2A_AGENT_URL set to {BACKEND_URL}")


## 14. Deploy frontend to Cloud Run

Uses the runtime-injection pattern — `API_BASE` and `NOVNC_URL` set at deploy time, no rebuild.

In [ ]:
sh(f"gcloud run deploy {FRONTEND_SERVICE} "
   f"--image={FRONTEND_IMAGE} --region={INFRA_REGION} "
   f"--set-env-vars=API_BASE={BACKEND_URL},NOVNC_URL={NOVNC_URL_VAL} "
   f"--memory=512Mi --cpu=1 --max-instances=3 "
   f"--allow-unauthenticated --quiet --project={PROJECT_ID}", capture=False)

FRONTEND_URL = sh(f"gcloud run services describe {FRONTEND_SERVICE} "
                  f"--region={INFRA_REGION} --format=value(status.url) --project={PROJECT_ID}")
print(f"\nFrontend URL: {FRONTEND_URL}")


## 15. Salesforce one-time login (human-in-the-loop)

The browser VM's Chrome session opened to your Salesforce login page when the container started. The Computer Use agent CAN type credentials, but the cleanest demo flow is to log in once yourself, then let Computer Use pick up the authenticated session.

**Open the noVNC URL printed below in a new browser tab**, log in with the Salesforce username + password from §1.3 / §9.2, navigate to the Sales app if prompted, then come back and run the next cell.

In [ ]:
print("Open this in a new browser tab:")
print(f"  {NOVNC_URL_VAL}")
print()
print("Steps inside noVNC:")
print("  1. Wait for the Salesforce login page to load")
print("  2. Enter your Salesforce username (email-format login)")
print("  3. Enter your password")
print("  4. If prompted, accept the browser device verification (one-time)")
print("  5. Land on the Sales app home page")
print("  6. Come back here and run the next cell")


In [ ]:
_ = input("Press Enter once you are logged into Salesforce in the noVNC tab... ")
print("OK — Computer Use can now drive the authenticated session.")


## 16. Smoke tests

Three quick checks confirming end-to-end wiring.

### 16.1 Backend health

In [ ]:
import urllib.request
with urllib.request.urlopen(f"{BACKEND_URL}/api/health", timeout=10) as r:
    body = r.read().decode()
print(f"  /api/health -> {r.status}: {body[:200]}")
assert r.status == 200, "Backend /api/health did not return 200"


### 16.2 Browser VM agent server health

In [ ]:
with urllib.request.urlopen(f"{BROWSER_AGENT_URL}/health", timeout=10) as r:
    body = r.read().decode()
print(f"  agent /health -> {r.status}: {body[:200]}")
assert r.status == 200, "Browser VM /health did not return 200"


### 16.3 Agent card (A2A discovery)

In [ ]:
with urllib.request.urlopen(f"{BACKEND_URL}/.well-known/agent.json", timeout=10) as r:
    card = json.loads(r.read())
print(json.dumps(card, indent=2))
assert card["url"] == BACKEND_URL, (
    f"A2A card URL ({card['url']}) does not match BACKEND_URL ({BACKEND_URL}). "
    "The §13 redeploy should have set A2A_AGENT_URL — check Cloud Run env vars."
)
print(f"\n[OK] A2A card URL matches deploy URL — safe to register with Gemini Enterprise.")


## 17. Deploy to Vertex AI Agent Engine

Wraps the ADK pipeline as an `AdkApp` and deploys it as a managed Agent Engine. Takes ~5–10 minutes. The deployed Agent Engine resource name is captured in `AGENT_ENGINE_RESOURCE` and reused in §18 when registering with Gemini Enterprise.

The repo's `deploy/agent_engine_deploy.py` hardcodes a project ID — we set the env vars first then patch in our project so the script is reusable.

In [ ]:
import sys

# Create staging bucket if it doesn't exist
try:
    sh(f"gcloud storage buckets describe {STAGING_BUCKET} --project={PROJECT_ID}", check=False)
    bucket_exists = sh(f"gcloud storage buckets list --filter=name:{STAGING_BUCKET[5:]} "
                       f"--format=value(name) --project={PROJECT_ID}", check=False)
    if not bucket_exists:
        sh(f"gcloud storage buckets create {STAGING_BUCKET} --location={INFRA_REGION} --project={PROJECT_ID}")
        print(f"  Created staging bucket: {STAGING_BUCKET}")
    else:
        print(f"  Staging bucket exists: {STAGING_BUCKET}")
except Exception as e:
    print(f"  Bucket setup error (continuing): {e}")

# Set env vars expected by agent_deploy/agent.py — these override the os.environ.setdefault() calls in that file
os.environ["GOOGLE_CLOUD_PROJECT"]  = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = MODEL_REGION
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"]     = MODEL_REGION
os.environ["BQ_DATASET"] = BQ_DATASET
os.environ["MODEL_PROVIDER"] = "claude"
os.environ["OPUS_MODEL"]   = OPUS_MODEL
os.environ["SONNET_MODEL"] = SONNET_MODEL
os.environ["HAIKU_MODEL"]  = HAIKU_MODEL
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
# Needed by agent_deploy/tools.py trigger_salesforce_opportunity (parameterized in PR #3)
os.environ["BACKEND_URL"] = BACKEND_URL
os.environ["NOVNC_URL"]   = NOVNC_URL_VAL

# Make the demo's agent_deploy package importable
sys.path.insert(0, DEMO_DIR)

# Re-import in case a prior cell already imported with stale env
import importlib
if "agent_deploy" in sys.modules:
    importlib.reload(sys.modules["agent_deploy"])

from agent_deploy.agent import root_agent  # noqa: E402
import vertexai
from vertexai import agent_engines

vertexai.init(project=PROJECT_ID, location=AGENT_ENGINE_LOCATION, staging_bucket=STAGING_BUCKET)

print("Wrapping ADK agent in AdkApp...")
app = agent_engines.AdkApp(agent=root_agent, enable_tracing=True)

# Quick local smoke
print("Local smoke test (1 message)...")
try:
    list(app.stream_query(user_id="notebook-smoke", message="hello"))
    print("  Local stream_query succeeded.")
except Exception as e:
    print(f"  Local smoke produced exception (often OK for streaming): {type(e).__name__}: {e}")

print("\nDeploying to Agent Engine (5-10 min) ...")
client = vertexai.Client(project=PROJECT_ID, location=AGENT_ENGINE_LOCATION)
remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": [
            "cloudpickle>=3.0.0",
            "pydantic>=2.0.0",
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-adk>=1.2.0",
            "anthropic[vertex]>=0.43.0",
            "google-cloud-bigquery>=3.27.0",
            "httpx>=0.27.0",
        ],
        "staging_bucket": STAGING_BUCKET,
        "display_name": "Deal Desk Agent",
        "description": "FSI Deal Desk pipeline — Claude on Vertex AI + ADK",
        "service_account": SA_EMAIL,
        "env_vars": {
            "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
            "PROJECT_ID":  PROJECT_ID,
            "REGION":      MODEL_REGION,
            "BQ_DATASET":  BQ_DATASET,
            "OPUS_MODEL":  OPUS_MODEL,
            "SONNET_MODEL": SONNET_MODEL,
            "HAIKU_MODEL": HAIKU_MODEL,
            "BACKEND_URL": BACKEND_URL,
            "NOVNC_URL":   NOVNC_URL_VAL,
        },
    },
)

AGENT_ENGINE_RESOURCE = str(remote_agent.api_resource)
print(f"\n[OK] Agent Engine deployed.")
print(f"  Resource name: {AGENT_ENGINE_RESOURCE}")
print()
print(f"  Save this for the Gemini Enterprise registration in §18:")
print(f"  AGENT_ENGINE_RESOURCE = \"{AGENT_ENGINE_RESOURCE}\"")


## 18. Register agent with Gemini Enterprise

Adds the deployed agent to your Gemini Enterprise engine so end-users can discover and invoke it from the Gemini Enterprise UI.

We register **two** integration paths so the agent works whichever way Gemini Enterprise wants to call it:

1. **A2A endpoint** — the public Cloud Run backend URL serving `/.well-known/agent.json` (the path Gemini Enterprise uses for streaming, multi-turn conversations)
2. **Agent Engine resource** — the Vertex AI Agent Engine resource captured in §17 (used when Gemini Enterprise prefers managed-runtime execution)

### 18.1 Confirm the engine ID and where to use the registered agent later

In [ ]:
print(f"Registering into Gemini Enterprise engine:")
print(f"  Project:     {PROJECT_ID}")
print(f"  Location:    {DISCOVERY_ENGINE_LOCATION}")
print(f"  Engine ID:   {GEMINI_ENTERPRISE_ENGINE_ID}")
print()
print(f"After registration completes, your end-users will find the agent at:")
print(f"  https://console.cloud.google.com/gen-app-builder/engines/{GEMINI_ENTERPRISE_ENGINE_ID}/agents?project={PROJECT_ID}")


### 18.2 Register the A2A endpoint

Posts the agent definition to the Discovery Engine `agents` collection on your engine. The agent card content from `/.well-known/agent.json` is included so Gemini Enterprise has the schema for skills, capabilities, and the deployed Agent Engine resource.

In [ ]:
import httpx
creds.refresh(google.auth.transport.requests.Request())
token = creds.token

# Pull the live agent card
with urllib.request.urlopen(f"{BACKEND_URL}/.well-known/agent.json", timeout=10) as r:
    live_card = json.loads(r.read())

# Build the registration payload. The Discovery Engine `agents.create` shape:
#   POST .../engines/{engine}/assistants/default_assistant/agents
# Each agent is identified by an `agent_id` you choose.
parent = (
    f"projects/{PROJECT_ID}/locations/{DISCOVERY_ENGINE_LOCATION}"
    f"/collections/default_collection/engines/{GEMINI_ENTERPRISE_ENGINE_ID}"
    f"/assistants/default_assistant"
)
agent_id = "deal-desk-agent"

agent_resource = {
    "displayName": "FSI Deal Desk Agent",
    "description": live_card["description"],
    "icon": {
        "uri": "https://www.gstatic.com/images/icons/material/system/2x/business_center_black_48dp.png"
    },
    # A2A endpoint
    "a2aAgentDefinition": {
        "jsonAgentCard": json.dumps(live_card),
    },
    # Vertex AI Agent Engine link — captured in §17
    "managedAgentDefinition": {
        "agentEngineRuntime": {
            "agentEngine": AGENT_ENGINE_RESOURCE,
        },
    },
}

url = (
    f"https://discoveryengine.googleapis.com/v1alpha/{parent}/agents"
    f"?agentId={agent_id}"
)

resp = httpx.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json=agent_resource,
    timeout=30,
)

if resp.status_code == 200:
    print(f"[OK] Agent registered.")
    print(json.dumps(resp.json(), indent=2)[:1500])
elif resp.status_code == 409:
    # Already exists — patch it instead
    print("Agent already exists — updating in place...")
    patch_url = f"https://discoveryengine.googleapis.com/v1alpha/{parent}/agents/{agent_id}"
    resp = httpx.patch(
        patch_url,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=agent_resource,
        timeout=30,
    )
    print(f"  PATCH status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2)[:1500])
else:
    print(f"[WARN] Registration returned {resp.status_code}")
    print(resp.text[:1500])
    print()
    print("If this is your first time using the v1alpha agents API, the resource "
          "shape may have changed. Check the latest spec at "
          "https://cloud.google.com/agentspace/docs/reference/rest")


### 18.3 Verify discoverability

In [ ]:
list_url = f"https://discoveryengine.googleapis.com/v1alpha/{parent}/agents"
resp = httpx.get(list_url, headers={"Authorization": f"Bearer {token}"}, timeout=15)
agents_list = resp.json().get("agents", [])
print(f"Engine {GEMINI_ENTERPRISE_ENGINE_ID} has {len(agents_list)} registered agent(s):")
for a in agents_list:
    print(f"  - {a.get('displayName')}  ({a.get('name')})")


## 19. Deployment summary

Everything is up. Bookmark these.

In [ ]:
from IPython.display import Markdown
Markdown(f"""
### Live URLs

| Resource | URL |
|---|---|
| **Frontend** (the demo UI) | {FRONTEND_URL} |
| **Backend** (FastAPI on Cloud Run) | {BACKEND_URL} |
| **Backend health** | {BACKEND_URL}/api/health |
| **A2A agent card** | {BACKEND_URL}/.well-known/agent.json |
| **noVNC** (browser VM live view) | {NOVNC_URL_VAL} |
| **Agent Engine resource** | `{AGENT_ENGINE_RESOURCE}` |
| **Gemini Enterprise discovery** | https://console.cloud.google.com/gen-app-builder/engines/{GEMINI_ENTERPRISE_ENGINE_ID}/agents?project={PROJECT_ID} |

### Try it

- **From the demo frontend** — open the Frontend URL above, pick a preset prompt, watch the multi-agent pipeline run
- **From Gemini Enterprise** — open your engine's chat surface and ask "Onboard new client: [name]. $250M AUM, long/short equity"
- **Direct A2A call** — POST to the backend root with `{{"jsonrpc":"2.0","method":"message/send","params":{{...}}}}`
- **Watch Salesforce automation live** — keep the noVNC tab open during a deal-pipeline run; the Salesforce agent will drive it

### When you're done, run §20 (teardown) to stop billing.
""")


## 20. Teardown — stop billing

> **Important:** the GCE browser VM bills 24/7. Run these cells when you're done with the demo to avoid surprise charges.

Each cell deletes one resource type. They can be run in any order.

### 20.1 Delete GCE instance + static IP + firewall rules

In [ ]:
sh(f"gcloud compute instances delete {GCE_INSTANCE} --zone={GCE_ZONE} --quiet --project={PROJECT_ID}", check=False)
sh(f"gcloud compute addresses delete {GCE_STATIC_IP} --region={INFRA_REGION} --quiet --project={PROJECT_ID}", check=False)
sh(f"gcloud compute firewall-rules delete {NOVNC_RULE} --quiet --project={PROJECT_ID}", check=False)
sh(f"gcloud compute firewall-rules delete {AGENT_RULE} --quiet --project={PROJECT_ID}", check=False)
print("GCE + firewall resources deleted.")


### 20.2 Delete Cloud Run services

In [ ]:
sh(f"gcloud run services delete {BACKEND_SERVICE} --region={INFRA_REGION} --quiet --project={PROJECT_ID}", check=False)
sh(f"gcloud run services delete {FRONTEND_SERVICE} --region={INFRA_REGION} --quiet --project={PROJECT_ID}", check=False)
print("Cloud Run services deleted.")


### 20.3 Delete Agent Engine deployment + Gemini Enterprise registration

In [ ]:
# Unregister from Gemini Enterprise
del_url = f"https://discoveryengine.googleapis.com/v1alpha/{parent}/agents/deal-desk-agent"
resp = httpx.delete(del_url, headers={"Authorization": f"Bearer {token}"}, timeout=15)
print(f"  Gemini Enterprise unregister: HTTP {resp.status_code}")

# Delete Agent Engine deployment
try:
    client.agent_engines.delete(name=AGENT_ENGINE_RESOURCE, force=True)
    print(f"  Agent Engine deleted: {AGENT_ENGINE_RESOURCE}")
except Exception as e:
    print(f"  Agent Engine delete failed (may already be gone): {e}")


### 20.4 Delete Artifact Registry images, BigQuery dataset, secrets, service account, staging bucket

Most expensive resources are gone after §20.1–20.3. The rest is housekeeping. Skip if you want to redeploy soon (these are reusable).

In [ ]:
# Artifact Registry repo (deletes all images)
sh(f"gcloud artifacts repositories delete {ARTIFACT_REPO} --location={INFRA_REGION} --quiet --project={PROJECT_ID}", check=False)

# BigQuery dataset
sh(f"bq rm -r -f --dataset {PROJECT_ID}:{BQ_DATASET}", check=False)

# Secrets
for s in [SECRET_AGENT, SECRET_SF_URL, SECRET_SF_USER, SECRET_SF_PASS]:
    sh(f"gcloud secrets delete {s} --quiet --project={PROJECT_ID}", check=False)

# Staging bucket (recursively)
sh(f"gcloud storage rm -r {STAGING_BUCKET} --quiet --project={PROJECT_ID}", check=False)

# Service account
sh(f"gcloud iam service-accounts delete {SA_EMAIL} --quiet --project={PROJECT_ID}", check=False)

print("\nAll Deal Desk resources deleted. APIs left enabled (no cost).")
